# Chạy lại Flora trên feature STAR (1s-Shift-GCN), NTU X-Sub

Mục đích:
1. Kiểm tra pipeline (feature `.npz`, split unseen, CLIP, semantics) cho ra số gần với bài báo: ZSL **85.8 / 61.5 / 80.7 / 64.1** (55/5, 48/12, 110/10, 96/24).
2. Ghi lại hai cách tính:
   - **best**: cách bài báo dùng. Cứ `eval_iter` iteration lại đánh giá trên test và giữ giá trị max, tức là chọn trên test.
   - **last**: độ chính xác ở lần đánh giá cuối. Đây là cách tính sạch.
3. Xem cấu trúc `.npz` và chỉ số nhãn (đánh từ 0), để loader của model NeSy dùng lại được.

Không sửa code Flora. Số *last* lấy bằng cách đọc `run.log`.

Bản chạy lại này lấy config STAR cho 55/5, còn 48/12, 110/10, 96/24 thì lấy siêu tham số từ config SynSE cùng split. Repo không có config STAR cho các split đó, nên lệch vài điểm là bình thường.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/BaoHuy/zeroshot'
FEAT_ARCHIVE = f'{DRIVE_DIR}/flora_skeleton_features.archive'  # cache để không phải tải lại
OUT_DIR = f'{DRIVE_DIR}/flora_repro'                          # log + bảng kết quả
FLORA = '/content/Flora'
EXTRACT = '/content/flora_features'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
if not os.path.exists(FLORA):
    !git clone -q https://github.com/cseeyangchen/Flora.git {FLORA}
!git -C {FLORA} rev-parse HEAD | tee {OUT_DIR}/flora_commit.txt
!pip -q install gdown transformers pyyaml pandas

# models/fm.py luôn import Shift-GCN, và Shift-GCN import CUDA op `shift_cuda`.
# Vì use_features=True nên Shift-GCN không bao giờ chạy, chỉ cần một module rỗng để import qua.
stub = f'{FLORA}/models/shiftgcn/Temporal_shift/cuda/shift_cuda.py'
with open(stub, 'w') as f:
    f.write('# stub: Shift-GCN CUDA op is never called when use_features=True\n')

!cd {FLORA} && python -c "from models.fm import MappingNet; print('import OK')"

## Tải feature đã trích sẵn

Link Google Drive lấy từ README của Flora. Nếu `gdown` báo lỗi quota: mở `https://drive.google.com/file/d/1wrxetP_ItfJfxAfNPLdCRC2FZ8Mb9Pa1/view`, chọn *Make a copy* vào Drive của bạn, rồi đổi tên/di chuyển file thành `FEAT_ARCHIVE` ở trên.

In [ ]:
import glob, shutil, subprocess
FEAT_ID = '1wrxetP_ItfJfxAfNPLdCRC2FZ8Mb9Pa1'
if not os.path.exists(FEAT_ARCHIVE):
    !gdown {FEAT_ID} -O {FEAT_ARCHIVE}
!ls -lh {FEAT_ARCHIVE}

def find_star():
    hits = glob.glob(f'{EXTRACT}/**/star_features', recursive=True)
    return hits[0] if hits else None

if find_star() is None:
    os.makedirs(EXTRACT, exist_ok=True)
    kind = subprocess.run(['file', '-b', FEAT_ARCHIVE], capture_output=True, text=True).stdout
    print('archive type:', kind)
    if 'Zip' in kind:
        subprocess.run(['unzip', '-q', FEAT_ARCHIVE, '-d', EXTRACT], check=True)
    elif '7-zip' in kind:
        subprocess.run(['7z', 'x', '-y', f'-o{EXTRACT}', FEAT_ARCHIVE], check=True, stdout=subprocess.DEVNULL)
    elif 'RAR' in kind:
        subprocess.run(['unrar', 'x', '-o+', FEAT_ARCHIVE, EXTRACT + '/'], check=True, stdout=subprocess.DEVNULL)
    else:  # tar / tar.gz
        subprocess.run(['tar', '-xf', FEAT_ARCHIVE, '-C', EXTRACT], check=True)

STAR_DIR = find_star()
assert STAR_DIR, 'Không thấy thư mục star_features sau khi giải nén'
os.makedirs(f'{FLORA}/skeleton_features', exist_ok=True)
link = f'{FLORA}/skeleton_features/star_features'
if not os.path.exists(link):
    os.symlink(STAR_DIR, link)
!ls -lh {link}/

## Kiểm tra `.npz` và chỉ số nhãn

Cần thấy:
- `train_label` chỉ chứa lớp seen.
- `zsl_label` đúng bằng tập unseen của Flora.
- `gzsl_label` chứa đủ cả hai.

Tên lớp unseen in ra để đối chiếu với `kg_rules.json`. Ở đó `id = nhãn + 1`.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, FLORA)
from utils.utils import task_definition

# task -> (config gốc để lấy siêu tham số theo split, ZSL top-1 bài báo báo cáo trên feature STAR)
SPLITS = {
    'ntu60_xsub_seen55_unseen5':    ('configs/star/ntu60_xsub_unseen5.yaml',    85.8),
    'ntu60_xsub_seen48_unseen12':   ('configs/synse/ntu60_xsub_unseen12.yaml',  61.5),
    'ntu120_xsub_seen110_unseen10': ('configs/synse/ntu120_xsub_unseen10.yaml', 80.7),
    'ntu120_xsub_seen96_unseen24':  ('configs/synse/ntu120_xsub_unseen24.yaml', 64.1),
}
names120 = pd.read_csv(f'{FLORA}/semantics/ntu120.csv')['label'].tolist()

for task in SPLITS:
    d = np.load(f'{link}/{task}.npz')
    print(f'=== {task}')
    for k in d.files:
        print(f'  {k:12s} {d[k].shape} {d[k].dtype}')
    _, unseen, seen, *_ = task_definition(task)
    tr, zl, gl = (set(np.unique(d[k]).tolist()) for k in ('train_label', 'zsl_label', 'gzsl_label'))
    print('  train ⊆ seen          :', tr <= set(seen), f'({len(tr)} lớp, min={min(tr)}, max={max(tr)})')
    print('  train ∩ unseen = ∅    :', not (tr & set(unseen)))
    print('  zsl == unseen         :', zl == set(unseen))
    print('  gzsl == seen ∪ unseen :', gl == set(seen) | set(unseen))
    print('  unseen:', [f'{u}/{u+1}:{names120[u]}' for u in unseen])

## Sinh config và chạy

Mỗi lần chạy: `align_iter=1000` iteration cho phase 1, rồi đến `total_iter=1200` iteration cho phase 2, đánh giá mỗi 10 iteration. Log ghi vào `OUT_DIR/<task>_<setting>_seed<k>/run.log`. Run nào đã xong sẽ bị bỏ qua, nên nếu Colab ngắt thì chạy lại cell là được.

Nên chạy **55/5 ZSL trước**. Nếu số khớp với bài báo thì chạy tiếp các split còn lại.

In [ ]:
import yaml, time
os.makedirs(f'{FLORA}/configs/repro', exist_ok=True)

def make_cfg(task, setting, seed=1):
    with open(f'{FLORA}/{SPLITS[task][0]}') as f:
        cfg = yaml.load(f, Loader=yaml.FullLoader)
    out = f'{OUT_DIR}/{task}_{setting}_seed{seed}'
    cfg.update(
        task_name=task,
        data_path=f'skeleton_features/star_features/{task}.npz',
        use_features=True,
        shiftgcn_checkpoint_path=None,  # không load Shift-GCN
        clip_checkpoint_path='openai/clip-vit-large-patch14-336',
        num_worker=2,
        setting=setting,
        seed=seed,
        out_dir=out,
    )
    path = f'{FLORA}/configs/repro/{task}_{setting}_seed{seed}.yaml'
    with open(path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)
    return path, out

def run(task, setting='ZSL', seed=1):
    path, out = make_cfg(task, setting, seed)
    log = f'{out}/run.log'
    if os.path.exists(log) and 'Best' in open(log).read():
        print(f'[skip] {task} {setting} seed{seed} đã xong')
        return
    shutil.rmtree(out, ignore_errors=True)  # logger mở file ở chế độ append, xóa để không lẫn run cũ
    t0 = time.time()
    p = subprocess.run(['python', 'main.py', '--config', path], cwd=FLORA, capture_output=True, text=True)
    print(f'[{task} {setting} seed{seed}] exit={p.returncode}  {(time.time()-t0)/60:.1f} phút')
    print('\n'.join(p.stdout.strip().splitlines()[-2:]))
    if p.returncode != 0:
        print(p.stderr[-3000:])

In [ ]:
# Bước 1: chỉ 55/5 ZSL. Kỳ vọng best ≈ 85.8
run('ntu60_xsub_seen55_unseen5', 'ZSL')

In [ ]:
# Bước 2: tất cả split, ZSL rồi GZSL (seed 1)
for setting in ['ZSL', 'GZSL']:
    for task in SPLITS:
        run(task, setting)

## Tổng hợp: best (cách tính của bài báo) và last (cách tính sạch)

- **ZSL**: `best` = max qua mọi lần đánh giá. `last` = lần đánh giá cuối. `last10` = trung bình ± độ lệch chuẩn của 10 lần đánh giá cuối, cho biết kết quả dao động bao nhiêu.
- **GZSL**: `best` là lần có H cao nhất trên test (Flora làm vậy), `last` là lần cuối.
- Cột GZSL của bài báo: điền tay từ bảng GZSL của Flora.

In [ ]:
import re
ZSL_RE = re.compile(r'Iter (\d+): ZSL Accuracy: ([\d.]+)%')
GZSL_RE = re.compile(r'Iter (\d+): Seen Accuracy: ([\d.]+)%\s+Unseen Accuracy: ([\d.]+)%\s+Harmonic Mean: ([\d.]+)%')

def read_log(task, setting, seed=1):
    log = f'{OUT_DIR}/{task}_{setting}_seed{seed}/run.log'
    if not os.path.exists(log):
        return None
    text = open(log).read()
    rx = ZSL_RE if setting == 'ZSL' else GZSL_RE
    rows = [tuple(float(x) for x in m.groups()) for m in rx.finditer(text)]
    return np.array(rows) if rows else None

zsl_rows, gzsl_rows = [], []
for task, (_, paper) in SPLITS.items():
    a = read_log(task, 'ZSL')
    if a is not None:
        acc = a[:, 1]
        zsl_rows.append(dict(split=task, paper=paper, best=acc.max(), best_iter=int(a[acc.argmax(), 0]),
                             last=acc[-1], last10_mean=acc[-10:].mean(), last10_std=acc[-10:].std(),
                             best_minus_last=acc.max() - acc[-1], n_evals=len(acc)))
    g = read_log(task, 'GZSL')
    if g is not None:
        b = g[g[:, 3].argmax()]
        l = g[-1]
        gzsl_rows.append(dict(split=task, best_S=b[1], best_U=b[2], best_H=b[3], best_iter=int(b[0]),
                              last_S=l[1], last_U=l[2], last_H=l[3], n_evals=len(g)))

zsl_df = pd.DataFrame(zsl_rows).round(2)
gzsl_df = pd.DataFrame(gzsl_rows).round(2)
zsl_df.to_csv(f'{OUT_DIR}/summary_zsl.csv', index=False)
gzsl_df.to_csv(f'{OUT_DIR}/summary_gzsl.csv', index=False)
display(zsl_df)
display(gzsl_df)

## Đọc kết quả

- **`best` lệch bài báo trong khoảng 1–2 điểm** → pipeline đúng. Số `last` là baseline sạch để so với model NeSy.
- **55/5 khớp nhưng các split khác lệch nhiều** → nhiều khả năng do mượn siêu tham số từ config SynSE (`max_length`, `topk`, `mixed_weights`). Pipeline vẫn đúng. Ghi rõ điều này trong luận văn.
- **Mọi split đều lệch xa** → kiểm tra feature (cell kiểm tra `.npz`), checkpoint CLIP và file semantics.
- **`best_minus_last` lớn** → kết quả dao động mạnh giữa các lần đánh giá, và số bài báo báo cáo bị đẩy lên vì chọn trên test. Dùng làm luận điểm khi so sánh.

Muốn đo độ ổn định thì chạy thêm seed: `run(task, 'ZSL', seed=2)`, rồi sửa `read_log` / vòng tổng hợp theo seed.